# 2 · Risk drivers: WoE, IV and chi-square

Every bin edge, rare-category mapping, clipping bound and feature-selection statistic is fitted on the training 60%. Validation and test labels do not enter these calculations. WoE is log(good distribution / bad distribution), with 0.5 additive smoothing; missing values have a separate bucket.

In [1]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
from IPython.display import display, Image, Markdown

# The executor supplies these paths. Interactive runs default to local exports.
ROOT = Path(os.environ.get('HOME_CREDIT_PROJECT_ROOT', str(Path.cwd()))).resolve()
if not (ROOT / 'src' / 'analysis.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.analysis import ensure_analysis, load_features, eda_tables

INPUT = Path(os.environ.get('HOME_CREDIT_FEATURES', str(ROOT / 'exports' / 'applicant_features.csv')))
OUTPUT = Path(os.environ.get('HOME_CREDIT_ANALYSIS', str(ROOT / 'outputs' / 'analysis')))
STATUS = os.environ.get('HOME_CREDIT_DATA_STATUS', 'real')
LGD = float(os.environ.get('HOME_CREDIT_LGD', '0.45'))
SEED = int(os.environ.get('HOME_CREDIT_SEED', '42'))
manifest = ensure_analysis(INPUT, OUTPUT, data_status=STATUS, lgd=LGD, seed=SEED)
get_ipython().run_line_magic('matplotlib', 'inline')
display(Markdown(f'**Data status: {manifest["data_status"].upper()}**. '
    + ('Synthetic pipeline validation; no Home Credit conclusions.' if STATUS != 'real'
       else 'Computed from the supplied accepted-cohort export.')))


**Data status: REAL**. Computed from the supplied accepted-cohort export.

In [2]:
iv = pd.read_csv(OUTPUT / 'iv_summary.csv')
woe = pd.read_csv(OUTPUT / 'woe_bins.csv')
chi = pd.read_csv(OUTPUT / 'chi_square_tests.csv')
display(iv.head(25))
display(chi.head(25))

,feature,iv,kind,fit_split,selected
0,ext_source_3,0.302745,numeric,train,True
1,ext_source_2,0.278189,numeric,train,True
2,ext_source_1,0.140194,numeric,train,True
3,employed_years,0.108026,numeric,train,True
4,occupation_type,0.081565,categorical,train,True
5,age_years,0.081330,numeric,train,True
6,credit_to_goods,0.066868,numeric,train,True
7,bureau_credit_to_debt_ratio,0.062719,numeric,train,True
8,cc_utilization_mean,0.061336,numeric,train,True
9,installment_late_rate,0.060420,numeric,train,True


,feature,chi_square,degrees_of_freedom,p_value,cramers_v,train_applicants,min_expected_count,expected_cells_below_5_pct,asymptotic_check,fit_split,p_value_bh,significant_bh_0_05
0,ext_source_1,1754.190019,5,0.000000e+00,0.097506,184506,1299.092387,0.000000,adequate,train,0.000000e+00,True
1,ext_source_2,3963.051176,5,0.000000e+00,0.146558,184506,33.341111,0.000000,adequate,train,0.000000e+00,True
2,ext_source_3,4326.972450,5,0.000000e+00,0.153139,184506,2363.343875,0.000000,adequate,train,0.000000e+00,True
3,employed_years,1436.782241,5,1.474050e-308,0.088245,184506,2440.843794,0.000000,adequate,train,2.174224e-307,True
4,occupation_type,1174.312743,14,5.765952e-242,0.079779,184506,100.507707,0.000000,adequate,train,6.803824e-241,True
5,age_years,1089.038661,4,1.799160e-234,0.076827,184506,2978.257292,0.000000,adequate,train,1.769174e-233,True
6,cc_utilization_mean,990.129319,4,4.916868e-213,0.073256,184506,836.353289,0.000000,adequate,train,4.144218e-212,True
7,credit_to_goods,988.827787,4,9.413313e-213,0.073207,184506,12.997382,0.000000,adequate,train,6.942318e-212,True
8,bureau_credit_to_debt_ratio,965.670193,5,1.624772e-206,0.072345,184506,1798.966863,0.000000,adequate,train,1.065128e-205,True
9,installment_late_rate,889.578712,3,1.612341e-192,0.069436,184506,760.790869,0.000000,adequate,train,9.512810e-192,True


## Read a driver

High IV indicates separation of good/bad distributions in the training sample. IV is an association screen, not a guarantee of future performance. Chi-square tests use Benjamini–Hochberg false-discovery correction and report Cramér's V. Sparse expected counts are flagged; significance in a large dataset can coexist with a small practical effect.

In [3]:
top_feature = iv.iloc[0].feature
display(Markdown(f'### Train-only bins: {top_feature}'))
display(woe[woe.feature == top_feature])
assert set(woe.fit_split.unique()) == {'train'}
assert set(iv.fit_split.unique()) == {'train'}

### Train-only bins: ext_source_3

,feature,kind,bin,bin_description,applicants,bad_count,good_count,bad_rate,bad_rate_ci_low,bad_rate_ci_high,good_distribution,bad_distribution,woe,iv_component,fit_split
69,ext_source_3,numeric,bin_1,"[-infinity, 0.3296550543128238)",29275,4762,24513,0.162664,0.158481,0.166936,0.144525,0.319674,-0.793847,0.139041,train
70,ext_source_3,numeric,bin_2,"[0.3296550543128238, 0.4740512892789932)",29441,2613,26828,0.088754,0.085559,0.092056,0.158174,0.175426,-0.103524,0.001786,train
71,ext_source_3,numeric,bin_3,"[0.4740512892789932, 0.5902333386185574)",29678,1736,27942,0.058495,0.055881,0.061222,0.164742,0.116559,0.345979,0.016670,train
72,ext_source_3,numeric,bin_4,"[0.5902333386185574, 0.6940926425266661)",29938,1360,28578,0.045427,0.043126,0.047845,0.168491,0.091321,0.612504,0.047267,train
73,ext_source_3,numeric,bin_5,"[0.6940926425266661, infinity)",29614,1006,28608,0.033970,0.031967,0.036095,0.168668,0.067559,0.914927,0.092507,train
74,ext_source_3,numeric,__MISSING__,__MISSING__,36560,3418,33142,0.093490,0.090549,0.096517,0.195400,0.229460,-0.160684,0.005473,train


In [4]:
display(pd.DataFrame([{'feature': name, 'kind': spec['kind'],
    'clip_low': spec.get('clip_low'), 'clip_high': spec.get('clip_high'),
    'numeric_edges': spec.get('edges'), 'rare_minimum': spec.get('minimum_count')}
    for name, spec in manifest['woe_specs'].items()]).head(30))

,feature,kind,clip_low,clip_high,numeric_edges,rare_minimum
0,age_years,numeric,22.617385,6.681451e+01,"[31.98631074606434, 39.44147843942505, 47.1184...",NaN
1,employed_years,numeric,0.303901,3.104449e+01,"[1.68104038329911, 3.4168377823408624, 5.91375...",NaN
2,is_employed_missing,numeric,0.000000,1.000000e+00,[0.0],NaN
3,income_amount,numeric,45000.000000,4.725000e+05,"[99000.0, 135000.0, 162000.0, 225000.0]",NaN
4,credit_amount,numeric,76410.000000,1.850544e+06,"[254700.0, 432661.5, 604152.0, 900000.0]",NaN
5,annuity_amount,numeric,6191.910000,7.020900e+04,"[14697.0, 21888.0, 28048.5, 37516.5]",NaN
6,goods_amount,numeric,67500.000000,1.800000e+06,"[225000.0, 378000.0, 522000.0, 823500.0]",NaN
7,credit_to_income,numeric,0.590080,1.304544e+01,"[1.8181818181818183, 2.764, 3.9032, 5.76761904...",NaN
8,annuity_to_income,numeric,0.039286,4.852859e-01,"[0.10350352941176465, 0.1438, 0.18566666666666...",NaN
9,credit_to_goods,numeric,1.000000,1.475200e+00,"[1.0, 1.079198675496689, 1.1452, 1.2112]",NaN


## Missing, rare and unseen values

Missing is explicitly included even when absent in train. Rare observed levels share a train-defined bucket. Unseen levels receive neutral WoE 0 when no rare bucket exists; otherwise they use the train rare-level bucket. Outlying future values are clipped using train 1st/99th percentiles before applying the persisted bins. The statistical tests are exploratory because continuous bins are data derived.